# 01 - Load and check the IBM AML dataset (HI-Small)

Manchester.Ai_IBM Z Datathon 2026 Project. Runs on IBM LinuxONE.

Goal: confirm the data loads, understand its shape, label balance and time range before building features.

In [ ]:
import platform, os
import pandas as pd
import numpy as np

# Evidence for judges: this is running on IBM Z (s390x)
print('Machine:', platform.machine())
print('Python :', platform.python_version())
print('pandas :', pd.__version__)

In [ ]:
# Data lives in ~/shared on the VM (Jupyter home is /home/ibm-user), outside the git repo
SHARED = os.path.expanduser('~/shared')
DATA_PATH = f'{SHARED}/HI-Small_Trans.csv'
assert os.path.exists(DATA_PATH), f'Not found: {DATA_PATH} - copy it into ~/shared on the VM first'
print(f'File size: {os.path.getsize(DATA_PATH) / 1e6:.0f} MB')

# Peek at the header before loading everything
pd.read_csv(DATA_PATH, nrows=5)

In [ ]:
%%time
# Load with compact dtypes to keep memory low on the shared VM.
# The CSV has two columns both called 'Account' (sender, receiver); pandas renames the second to 'Account.1'.
df = pd.read_csv(
    DATA_PATH,
    dtype={
        'From Bank': 'int32', 'Account': 'string',
        'To Bank': 'int32', 'Account.1': 'string',
        'Amount Received': 'float64', 'Receiving Currency': 'category',
        'Amount Paid': 'float64', 'Payment Currency': 'category',
        'Payment Format': 'category', 'Is Laundering': 'int8',
    },
)

df = df.rename(columns={
    'From Bank': 'from_bank', 'Account': 'from_account',
    'To Bank': 'to_bank', 'Account.1': 'to_account',
    'Amount Received': 'amount_received', 'Receiving Currency': 'receiving_currency',
    'Amount Paid': 'amount_paid', 'Payment Currency': 'payment_currency',
    'Payment Format': 'payment_format', 'Is Laundering': 'is_laundering',
    'Timestamp': 'timestamp',
})
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y/%m/%d %H:%M')

# Account numbers are only unique within a bank, so build a global ID
df['sender'] = df['from_bank'].astype(str) + '_' + df['from_account'].astype(str)
df['receiver'] = df['to_bank'].astype(str) + '_' + df['to_account'].astype(str)

# Sort by time - every later step depends on this (no future data leaking into features)
df = df.sort_values('timestamp', kind='stable').reset_index(drop=True)

print(df.shape)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

In [ ]:
# Label balance - expect a very small laundering share (heavily imbalanced)
counts = df['is_laundering'].value_counts()
print(counts)
print(f"Laundering share: {df['is_laundering'].mean():.4%}")

In [ ]:
# Time range and volume per day - decides where we split train / test
print('From:', df['timestamp'].min())
print('To  :', df['timestamp'].max())
daily = df.groupby(df['timestamp'].dt.date).agg(
    transactions=('is_laundering', 'size'),
    laundering=('is_laundering', 'sum'),
)
daily

In [ ]:
# Accounts involved
n_accounts = pd.unique(pd.concat([df['sender'], df['receiver']])).size
mule_receivers = df.loc[df['is_laundering'] == 1, 'receiver'].nunique()
print(f'Unique accounts: {n_accounts:,}')
print(f'Accounts that received laundered money: {mule_receivers:,}')

In [ ]:
# How laundering differs by payment format and currency
print(df.groupby('payment_format', observed=True)['is_laundering'].agg(['size', 'sum', 'mean']))
print()
print(df.groupby('payment_currency', observed=True)['is_laundering'].agg(['size', 'sum', 'mean']))

In [ ]:
# Self-transfers (same sender and receiver) - check how many before deciding whether to drop them
self_tx = (df['sender'] == df['receiver'])
print(f'Self-transfers: {self_tx.sum():,} ({self_tx.mean():.1%})')
print(f"Laundering among self-transfers: {df.loc[self_tx, 'is_laundering'].sum():,}")

In [ ]:
# Save a compact copy so later notebooks load in seconds instead of re-parsing the CSV
# (falls back to pickle if pyarrow isn't installed in the container)
try:
    df.to_parquet(f'{SHARED}/hi_small_clean.parquet', index=False)
    print('Saved shared/hi_small_clean.parquet')
except ImportError:
    df.to_pickle(f'{SHARED}/hi_small_clean.pkl')
    print('pyarrow missing - saved shared/hi_small_clean.pkl instead')